# S1-1 · GPT 的结构

`vendor/nanoGPT/model.py` 一共 330 行，定义了一整个 GPT。

这一节只看**骨架**：模型由哪几块组成，数据怎么从一串 token 流到一串分数。注意力机制单独放到 S1-2。

> 旁边用 VS Code 打开 `vendor/nanoGPT/model.py`，对照着看行号。

In [ ]:
import pathlib
import sys

# 依次尝试：notebook 自身位置（VS Code 提供）→ 当前目录 → 逐级向上
cands = []
nb_file = globals().get("__vsc_ipynb_file__")
if nb_file:
    cands.append(pathlib.Path(nb_file).parent)
cands += [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]

ROOT = next((c for c in cands if (c / "vendor" / "nanoGPT" / "model.py").exists()), None)
assert ROOT is not None, (
    "没找到 vendor/nanoGPT/model.py。\n"
    "请确认已 clone 到项目根目录下（vendor/nanoGPT），或把工作目录切到项目根再运行。"
)

SRC = ROOT / "vendor" / "nanoGPT" / "model.py"
print("项目根 :", ROOT)
print("源文件 :", SRC)
print("行数   :", len(SRC.read_text(encoding="utf-8").splitlines()))

sys.path.insert(0, str(SRC.parent))
import model as nanogpt

import torch

## 一、GPTConfig：六个数字定义一个模型

`model.py` 第 108–116 行：

```python
@dataclass
class GPTConfig:
    block_size: int = 1024      # 上下文长度上限
    vocab_size: int = 50304     # 词表大小
    n_layer: int = 12           # 多少层
    n_head: int = 12            # 多少个注意力头
    n_embd: int = 768           # 向量宽度
    dropout: float = 0.0
    bias: bool = True
```

**整个模型的结构信息，就这六个数。** 后面所有形状都能从它们推出来。

In [ ]:
cfg = nanogpt.GPTConfig()
for k, v in vars(cfg).items():
    print(f"  {k:<12} = {v}")

print("\n由它推出的关键形状：")
print(f"  每个 token 的向量长度 = n_embd = {cfg.n_embd}")
print(f"  每个头的维度         = n_embd / n_head = {cfg.n_embd // cfg.n_head}")
print(f"  MLP 中间层宽度       = 4 * n_embd = {4 * cfg.n_embd}")

In [ ]:
m = nanogpt.GPT(cfg)
print("\n参数量：%.2fM" % (m.get_num_params() / 1e6))

## 二、`__init__`：五个组件

`model.py` 第 126–133 行。

| 组件 | 类型 | 形状 | 作用 |
|---|---|---|---|
| `wte` | Embedding | `(vocab_size, n_embd)` | token 序号 → 向量 |
| `wpe` | Embedding | `(block_size, n_embd)` | 位置序号 → 向量 |
| `drop` | Dropout | — | 正则化 |
| `h` | ModuleList | `n_layer` 个 Block | 主体 |
| `ln_f` | LayerNorm | — | 收尾归一化 |
| `lm_head` | Linear | `(vocab_size, n_embd)` | 向量 → 每个词的分数 |

注意 `lm_head` 不在 `transformer` 里，它是 GPT 的直接属性。

In [ ]:
print("transformer 的子模块：")
for name, mod in m.transformer.named_children():
    print(f"  {name:<6} {type(mod).__name__}")

print("\n各参数的形状：")
for name, p in m.named_parameters():
    if "h.1" in name or "h.2" in name:      # 只看前两层，后面的完全重复
        continue
    print(f"  {name:<28} {tuple(p.shape)}")

In [ ]:
print("层数检查：", len(m.transformer.h), "层")
print("第 0 层：")
print(m.transformer.h[0])

### 权重绑定（第 138 行）

```python
self.transformer.wte.weight = self.lm_head.weight
```

`wte` 和 `lm_head` **共用同一块参数**。

- `wte`：token → 向量（查表）
- `lm_head`：向量 → token 分数（反向查表）

两者都在描述「每个词对应哪个向量」，所以共用矩阵是合理的，还能省掉 `vocab_size × n_embd` 个参数（GPT-2 上是 50304 × 768 ≈ 3860 万）。

In [ ]:
print("wte 和 lm_head 是同一个张量吗：", m.transformer.wte.weight is m.lm_head.weight)
print("形状：", tuple(m.transformer.wte.weight.shape))
print("贡献了全部参数的 %.1f%%" % (100 * m.transformer.wte.weight.numel() / sum(p.numel() for p in m.parameters())))

## 三、`forward`：数据怎么流

第 170–193 行，去掉分支后只有五步：

```python
tok_emb = wte(idx)          # (b, t)      → (b, t, n_embd)
pos_emb = wpe(pos)          # (t,)        → (t, n_embd)
x = drop(tok_emb + pos_emb) # 相加 → (b, t, n_embd)
for block in h: x = block(x)   # 形状不变
x = ln_f(x)
logits = lm_head(x)         # → (b, t, vocab_size)
```

**关键观察**：`x` 从进第一个 Block 到出最后一个 Block，形状始终是 `(b, t, n_embd)`。这就是为什么能堆任意多层。

In [ ]:
# 用一个小配置，手工重走一遍 forward，把每步形状打出来
small = nanogpt.GPTConfig(block_size=16, vocab_size=11, n_layer=2, n_head=2, n_embd=8, dropout=0.0)
mm = nanogpt.GPT(small)

idx = torch.randint(0, small.vocab_size, (2, 5))     # b=2, t=5
print("idx          ", tuple(idx.shape))

tok = mm.transformer.wte(idx)
print("tok_emb      ", tuple(tok.shape), " ← 查表，每个序号变成长度 8 的向量")

pos = torch.arange(idx.size(1))
print("pos          ", tuple(pos.shape), " ← [0,1,2,3,4]")

pos_emb = mm.transformer.wpe(pos)
print("pos_emb      ", tuple(pos_emb.shape), " ← 注意：没有 batch 维")

x = tok + pos_emb
print("tok + pos    ", tuple(x.shape), " ← 广播把 (5,8) 复制到两个 batch")

for i, blk in enumerate(mm.transformer.h):
    x = blk(x)
    print(f"block {i} 之后 ", tuple(x.shape), " ← 形状没变")

x = mm.transformer.ln_f(x)
print("ln_f 之后    ", tuple(x.shape))

logits = mm.lm_head(x)
print("logits       ", tuple(logits.shape), " ← 每个位置一个词表大小的分数")

In [ ]:
# 验证：手工这一遍和直接调用模型结果一致
logits_direct, _ = mm(idx)
print("与 model(idx) 一致：", torch.allclose(logits, logits_direct))

## 四、为什么位置编码可以直接加

`pos_emb` 形状是 `(t, 8)`，`tok_emb` 是 `(2, t, 8)`，两个不一样却可以相加——因为**广播**（B1 那条规则）：

```
   2  t  8
      t  8      ← 左边补 1
   -------
   ✅  ✅ ✅
```

第一个维度 `2 vs 1` 可广播，其余相等。结果就是每个 batch 都加上同一份位置编码。

**为什么不直接加一个整数位置？** 因为位置和词向量得在同一个空间里才谈得上“相加有意义”。用 embedding 学出来的位置向量，模型可以自己决定“位置信息该以什么形式混进内容里”。

In [ ]:
p = torch.randn(5, 8)          # (t, C)
c = torch.randn(2, 5, 8)       # (b, t, C)
print("相加结果：", tuple((c + p).shape))
print("两个 batch 加的位置编码相同：", torch.allclose((c + p)[0] - c[0], (c + p)[1] - c[1]))

## 五、Block：只有三行有效代码

第 94–106 行：

```python
def forward(self, x):
    x = x + self.attn(self.ln_1(x))    # 注意力 + 残差
    x = x + self.mlp(self.ln_2(x))     # 前馈   + 残差
    return x
```

两个 `x +` 是**残差连接**：把输入原样加回输出。作用是让梯度有一条直通车的路，深层网络才训得动。

`ln_1(x)` 在进子层**之前**做归一化，这叫 **Pre-LN**（GPT-2 之后的标配）。老版本是 Post-LN（子层之后归一化），深层时不稳定。

In [ ]:
import inspect
print(inspect.getsource(nanogpt.Block))

In [ ]:
# 手动验证：Block 就是把两步残差加起来
blk = mm.transformer.h[0]
x = torch.randn(2, 5, 8)

manual = x + blk.attn(blk.ln_1(x))
manual = manual + blk.mlp(blk.ln_2(manual))

print("手动结果和 blk(x) 一致：", torch.allclose(blk(x), manual, atol=1e-6))

## 六、训练 vs 推理：两条分支

第 184–191 行：

```python
if targets is not None:
    logits = self.lm_head(x)                                    # 所有位置
    loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ...)
else:
    logits = self.lm_head(x[:, [-1], :])                        # 只要最后一个位置
```

**训练时**：每个位置都要预测下一个词，所以全算，然后摊平成 `(b*t, vocab)` 喂给交叉熵——就是 B5 那件事，只是批量更大。

**推理时**：你只要下一个词，前面位置的 logits 算出来也没人用，所以只取最后一个位置。

In [ ]:
tgt = torch.randint(0, small.vocab_size, (2, 5))
logits_t, loss = mm(idx, tgt)
print("训练模式 logits:", tuple(logits_t.shape), " loss:", round(loss.item(), 3))

logits_i, loss_i = mm(idx)
print("推理模式 logits:", tuple(logits_i.shape), " loss:", loss_i)

# 交叉熵那一步在做什么
flat = logits_t.view(-1, logits_t.size(-1))
print("\n摊平后：", tuple(flat.shape), " 对应 ", tgt.view(-1).shape[0], " 个预测")

## 七、留给你回答的三个问题

1. `pos_emb` 形状 `(t, n_embd)`，`tok_emb` 是 `(b, t, n_embd)`，为什么能相加？
2. 训练时 `logits` 是 `(b, t, vocab)`，为什么喂给 `cross_entropy` 前要 `.view(-1, vocab)`？
3. 推理时为什么只算最后一个位置，而不是像训练那样全算？

答完这三题，再进 S1-2 看注意力。